# Градиентный бустинг. Улучшение модели с занятий. Домашка
датасет про калифорнийское жилье. класс как на паре. вариация дз 1 - 4 пункта добавить

In [92]:
!pip install catboost
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.tree import DecisionTreeRegressor

import xgboost as xgb
import lightgbm as lgb
import catboost as cb

sns.set_theme(style="whitegrid")

In [93]:
data = fetch_california_housing()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target
print(X.info())
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20640 entries, 0 to 20639
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   MedInc      20640 non-null  float64
 1   HouseAge    20640 non-null  float64
 2   AveRooms    20640 non-null  float64
 3   AveBedrms   20640 non-null  float64
 4   Population  20640 non-null  float64
 5   AveOccup    20640 non-null  float64
 6   Latitude    20640 non-null  float64
 7   Longitude   20640 non-null  float64
dtypes: float64(8)
memory usage: 1.3 MB
None


In [94]:
class MyBoost:
  def __init__(self, n_estimators=400, learning_rate=0.05, max_depth=7, seed=42, subsample = 1.0, colsample_bytree = 1.0, cat_smoothing = 10.0) -> None:
      self.n_estimators = n_estimators
      self.learning_rate = learning_rate
      self.max_depth = max_depth
      self.seed = seed
      self.subsample = subsample
      self.colsample_bytree = colsample_bytree
      self.cat_smoothing = cat_smoothing
      self.global_mean_ = 0
      self.trees = []
      self.feature_indices_list = []
      self.feature_importances_ = None
      self.cat_encodings_ = {}

  # выбор рандомной части записей для обучения
  def custom_subsample(self, X, idx):
    n_samples = X.shape[0]
    n_subsample = int(n_samples * self.subsample)
    indices = np.random.RandomState(self.seed + idx).choice(n_samples, n_subsample, replace=False)
    return indices

  # выбор рандомных признаков для обучния дерева
  def custom_colsample_bytree(self, X, idx):
    n_features = X.shape[1]
    n_sample_features = int(self.colsample_bytree * n_features)
    feature_indices = np.random.RandomState(self.seed + 4242 + idx).choice(n_features, n_sample_features, replace=False)
    return feature_indices

  # кодирование категориальных со smoothing
  def smoothing(self, X, y, fit):
    X = X.copy()

    categorical_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()
    if not categorical_cols:
      return X
    if fit == True:
      global_mean = y.mean()
      self.global_mean_ = global_mean
    else:
      global_mean = self.global_mean_

    for col in categorical_cols:
          if fit==True:
              categories = X[col].unique()
              encoding = {}

              for category in categories:
                  mask = (X[col] == category)
                  count = mask.sum()
                  sum_target = y[mask].sum()
                  smooth_value = (sum_target + global_mean * self.cat_smoothing) / (count + self.cat_smoothing)
                  encoding[category] = smooth_value
              self.cat_encodings_[col] = encoding
              X[col] = X[col].map(encoding).astype(float)
          else:
              X[col] = X[col].map(self.cat_encodings_[col]).fillna(self.global_mean_).astype(float)
    return X


  def fit(self, X, y):
    X = self.smoothing(X, y, fit=True)


    self.initial_leaf = y.mean()
    predictions = np.zeros(len(y)) + self.initial_leaf

    for idx in range(self.n_estimators):
      indices = self.custom_subsample(X, idx)

      X_sample = X.iloc[indices]
      y_sample = y[indices]
      pred_sample = predictions[indices]

      feature_indices = self.custom_colsample_bytree(X, idx)
      self.feature_indices_list.append(feature_indices)
      X_sample = X_sample.iloc[:, feature_indices]

      antigrad = y_sample - pred_sample
      tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=(self.seed + idx), criterion="friedman_mse")

      tree.fit(X_sample, antigrad)
      self.trees.append(tree)

      predictions += tree.predict(X.iloc[:, feature_indices]) * self.learning_rate

    total_importances = np.zeros(X.shape[1])
    for tree, feature_indices in zip(self.trees, self.feature_indices_list):
      tree_importances = tree.feature_importances_
      for position, feature_index in enumerate(feature_indices):
        total_importances[feature_index] += tree_importances[position] * self.learning_rate

    total_importances /= self.n_estimators
    total_importances /= total_importances.sum()
    self.feature_importances_ = total_importances




  def predict(self, samples):
    predictions = np.zeros(len(samples)) + self.initial_leaf
    #for i in range(self.n_estimators):
      #predictions += self.learning_rate * self.trees[i].predict(samples)

    samples = self.smoothing(samples, None, fit=False)
    for i, tree in enumerate(self.trees):
      feature_indices = self.feature_indices_list[i]
      predictions += self.learning_rate * tree.predict(samples.iloc[:, feature_indices])

    return predictions

# ПРОВЕРКА SUBSAMPLE

In [99]:
subsamples = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

for i in  subsamples:
  start_time = time.time()
  model = MyBoost(subsample = i)
  model.fit(X_train, y_train)
  learning_time = time.time() - start_time
  y_pred = model.predict(X_test)
  print("Subsample: ", i)
  print(f"R2: {r2_score(y_test, y_pred):.4f}" )
  print(f"Learning time: {learning_time:.3f}\n\n")

Subsample:  0.5
R2: 0.8437
Learning time: 27.592


Subsample:  0.6
R2: 0.8452
Learning time: 32.416


Subsample:  0.7
R2: 0.8457
Learning time: 37.334


Subsample:  0.8
R2: 0.8439
Learning time: 41.903


Subsample:  0.9
R2: 0.8458
Learning time: 47.284


Subsample:  1.0
R2: 0.8415
Learning time: 55.468




в данном случае лучшие метрики при subsample 0.7 или 0.9. время обучения моделей с мин и макс значениями параметра отличается буквально в 2 раза.

# ПРОВЕРКА COLSAMPLE_BYTREE

In [100]:
colsample_bytree_list = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0]

# выбрал subsample который учился быстрее всех
for i in colsample_bytree_list:
  start_time = time.time()
  model = MyBoost(subsample = 0.5, colsample_bytree= i)
  model.fit(X_train, y_train)
  learning_time = time.time() - start_time
  y_pred = model.predict(X_test)
  print("Colsample bytree: ", i)
  print(f"R2: {r2_score(y_test, y_pred):.4f}" )
  print(f"Learning time: {learning_time:.3f}\n\n")

Colsample bytree:  0.5
R2: 0.8423
Learning time: 16.208


Colsample bytree:  0.6
R2: 0.8423
Learning time: 17.880


Colsample bytree:  0.7
R2: 0.8457
Learning time: 22.104


Colsample bytree:  0.8
R2: 0.8446
Learning time: 23.803


Colsample bytree:  0.9
R2: 0.8459
Learning time: 26.217


Colsample bytree:  1.0
R2: 0.8437
Learning time: 33.698




для выбранного subsample=0.5 лучшими значениями параметра colsample_bytree стали 0.7 и 0.9. модели стали обучаться еще быстрее при более низких colsample_bytree

интересное наблюдение:

if self.colsample_bytree >= 0.99:

      return np.arange(n_features)

когда я добавил в класс colsample_bytree и функцию выбора без if который указан выше метрика для случая когда subsample = 0.5 и colsample_bytree = 1.0 поменялась. исходя из этого можно сделать что порядок фичей влияет на дерево. ибо сначала было значение 0.8448 а стало 0.8437

# ПРОВЕРКА РАБОТОСПОСБНОСТИ FEATURE_IMPORTANCES_

In [97]:
model = MyBoost(subsample = 0.5, colsample_bytree= 0.5)
start_time = time.time()
model.fit(X_train, y_train)
learning_time = time.time() - start_time
y_pred = model.predict(X_test)
print(f"R2: {r2_score(y_test, y_pred):.4f}" )
print(f"Learning time: {learning_time:.3f}\n\n")

feature_iportances = model.feature_importances_
print("Feature importances:", model.feature_importances_)

sorted_indeces = np.argsort(model.feature_importances_)[::-1]
for i in sorted_indeces:
    print(f"{X.columns[i]}: {model.feature_importances_[i]:.4f}")

R2: 0.8423
Learning time: 14.922


Feature importances: [0.15535031 0.07665684 0.14207792 0.10560013 0.10301522 0.14385835
 0.13793949 0.13550173]
MedInc: 0.1554
AveOccup: 0.1439
AveRooms: 0.1421
Latitude: 0.1379
Longitude: 0.1355
AveBedrms: 0.1056
Population: 0.1030
HouseAge: 0.0767


сумма долей как и полагает быть равна единице.

# ПОДДЕРЖКА КАТЕГОРИАЛЬНЫХ ПРИЗНАКОВ
в датасете категориальных признаков нет. дополнительный параметр Latitude_over_mean вероятно бесполезен. это исключительно для вроверки кодируемости категориальных признаков

In [104]:
X_train_cat = X_train.copy()
X_test_cat = X_test.copy()

mean_latitude = X_train_cat['Latitude'].mean()
X_train_cat['Latitude_over_mean'] = (X_train_cat['Latitude'] > mean_latitude).map({True: 'over', False: 'under'})
X_test_cat['Latitude_over_mean'] = (X_train_cat['Latitude'] > mean_latitude).map({True: 'over', False: 'under'})

print("Latitude_over_median cats:", X_train_cat['Latitude_over_mean'].unique())

model = MyBoost(subsample=0.5, colsample_bytree=0.5, cat_smoothing=10.0)
model.fit(X_train_cat, y_train)
preds = model.predict(X_test_cat)

print(f"\nR2: {r2_score(y_test, preds):.4f}")
print(f"Кодировки: {model.cat_encodings_}")

Latitude_over_median cats: ['under' 'over']

R2: 0.8118
Кодировки: {'Latitude_over_mean': {'under': np.float64(2.176558138202687), 'over': np.float64(1.9349835646562072)}}


как можно видеть признак был закодирован